In [1]:
import pandas as pd

In [2]:
from google.colab import files
uploaded = files.upload()

Saving train.csv to train.csv


In [3]:
df = pd.read_csv('train.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
#информация об основных типах данных по столбцам
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [5]:
#подсчет пропусков. Наибольшее число пропусков по стобцу Cabin
df.isna().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


In [6]:
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [7]:
df['Age'].mode() #мода

,Age
0,24.0


Данные по возрасту заполнены по 714 пассажирам. Наименьший возраст 0.420000 лет, максимальны возврат 80.000000. Средний возраст около 30 лет. Наиболее часто всречаемый возраст 24 года

In [8]:
df['Pclass'].unique() #смотрим классы

array([3, 1, 2])

In [9]:
#смотрим распределение количества пассажирова и  выживших по классам
grouped_sv = df.groupby("Pclass").agg({
          "PassengerId": "count",
          "Survived": "sum"
      })

grouped_sv

,PassengerId,Survived
Pclass,,
1,216,136
2,184,87
3,491,119


In [10]:
#считаем процент выживаемости у каждого класса пассажиров
grouped_sv['share_survived_%'] = grouped_sv['Survived']/grouped_sv['PassengerId']*100
grouped_sv

,PassengerId,Survived,share_survived_%
Pclass,,,
1,216,136,62.962963
2,184,87,47.282609
3,491,119,24.236253


больший процент выживаемости в 1 классе и по количеству в том числе

In [11]:
import re


def extract_accurate_first_name(full_name: str) -> str | None:
    if not isinstance(full_name, str):
        return None

    match_parentheses = re.search(r"\(([^)]+)\)", full_name)
    if match_parentheses:
        inner_text = match_parentheses.group(1).strip()
        return inner_text.split()[0]

    parts = full_name.split(",", 1)
    if len(parts) < 2:
        return None

    after_comma = parts[1].strip()


    name_part = re.sub(r'^\s*\b[A-Za-z]+\.\s*', '', after_comma)

    if not name_part:
        return None

    return name_part.split()[0]

df["first_name"] = df["Name"].apply(extract_accurate_first_name)

female_mode = df[df["Sex"] == "female"]["first_name"].mode()
male_mode = df[df["Sex"] == "male"]["first_name"].mode()

print("DataFrame с извлеченными именами:")
print(df[["Name", "Sex", "first_name"]])
print("\nСамое популярное женское имя:", female_mode.iloc[0] if not female_mode.empty else "Нет данных")
print("Самое популярное мужское имя:", male_mode.iloc[0] if not male_mode.empty else "Нет данных")


DataFrame с извлеченными именами:
                                                  Name     Sex first_name
0                              Braund, Mr. Owen Harris    male       Owen
1    Cumings, Mrs. John Bradley (Florence Briggs Th...  female   Florence
2                               Heikkinen, Miss. Laina  female      Laina
3         Futrelle, Mrs. Jacques Heath (Lily May Peel)  female       Lily
4                             Allen, Mr. William Henry    male    William
..                                                 ...     ...        ...
886                              Montvila, Rev. Juozas    male     Juozas
887                       Graham, Miss. Margaret Edith  female   Margaret
888           Johnston, Miss. Catherine Helen "Carrie"  female  Catherine
889                              Behr, Mr. Karl Howell    male       Karl
890                                Dooley, Mr. Patrick    male    Patrick

[891 rows x 3 columns]

Самое популярное женское имя: Anna
Самое популярное м

Самое популярное женское имя: Anna
Самое популярное мужское имя: William

In [12]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,first_name
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,Owen
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,Florence
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,Laina
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,Lily
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,William


In [13]:
tmp = (
    df.groupby(["Pclass", "Sex"])["first_name"]
      .value_counts()
      .groupby(level=[0, 1])
      .head(1)
      .rename("count")
      .reset_index()
)
tmp

,Pclass,Sex,first_name,count
0,1,female,Elizabeth,5
1,1,male,William,11
2,2,female,Elizabeth,5
3,2,male,William,9
4,3,female,Anna,9
5,3,male,William,15


In [16]:
#часть таблицы с пассажирами, возраст которых больше 44 лет
df_more_44 = df[df['Age'] > 44]
df_more_44.info()
df_more_44.head()

<class 'pandas.core.frame.DataFrame'>
Index: 115 entries, 6 to 879
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  115 non-null    int64  
 1   Survived     115 non-null    int64  
 2   Pclass       115 non-null    int64  
 3   Name         115 non-null    object 
 4   Sex          115 non-null    object 
 5   Age          115 non-null    float64
 6   SibSp        115 non-null    int64  
 7   Parch        115 non-null    int64  
 8   Ticket       115 non-null    object 
 9   Fare         115 non-null    float64
 10  Cabin        58 non-null     object 
 11  Embarked     114 non-null    object 
 12  first_name   115 non-null    object 
dtypes: float64(2), int64(5), object(6)
memory usage: 12.6+ KB


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,first_name
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S,Timothy
11,12,1,1,"Bonnell, Miss. Elizabeth",female,58.0,0,0,113783,26.5500,C103,S,Elizabeth
15,16,1,2,"Hewlett, Mrs. (Mary D Kingcome)",female,55.0,0,0,248706,16.0000,NaN,S,Mary
33,34,0,2,"Wheadon, Mr. Edward H",male,66.0,0,0,C.A. 24579,10.5000,NaN,S,Edward
52,53,1,1,"Harper, Mrs. Henry Sleeper (Myna Haxtun)",female,49.0,1,0,PC 17572,76.7292,D33,C,Myna


115 пассажиров старше 44 лет

In [21]:
mask = (df["Age"] < 44) & (df["Sex"] == "male")
df_less_44 = df[mask]
df_less_44.info()
df_less_44.head(10)

<class 'pandas.core.frame.DataFrame'>
Index: 368 entries, 0 to 890
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  368 non-null    int64  
 1   Survived     368 non-null    int64  
 2   Pclass       368 non-null    int64  
 3   Name         368 non-null    object 
 4   Sex          368 non-null    object 
 5   Age          368 non-null    float64
 6   SibSp        368 non-null    int64  
 7   Parch        368 non-null    int64  
 8   Ticket       368 non-null    object 
 9   Fare         368 non-null    float64
 10  Cabin        56 non-null     object 
 11  Embarked     368 non-null    object 
 12  first_name   368 non-null    object 
dtypes: float64(2), int64(5), object(6)
memory usage: 40.2+ KB


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,first_name
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.250,NaN,S,Owen
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.050,NaN,S,William
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.075,NaN,S,Gosta
12,13,0,3,"Saundercock, Mr. William Henry",male,20.0,0,0,A/5. 2151,8.050,NaN,S,William
13,14,0,3,"Andersson, Mr. Anders Johan",male,39.0,1,5,347082,31.275,NaN,S,Anders
16,17,0,3,"Rice, Master. Eugene",male,2.0,4,1,382652,29.125,NaN,Q,Eugene
20,21,0,2,"Fynney, Mr. Joseph J",male,35.0,0,0,239865,26.000,NaN,S,Joseph
21,22,1,2,"Beesley, Mr. Lawrence",male,34.0,0,0,248698,13.000,D56,S,Lawrence
23,24,1,1,"Sloper, Mr. William Thompson",male,28.0,0,0,113788,35.500,A6,S,William
27,28,0,1,"Fortune, Mr. Charles Alexander",male,19.0,3,2,19950,263.000,C23 C25 C27,S,Charles


368 пассажиров мужского пола и младше 44 лет

In [22]:

mask = (df['Cabin'].value_counts() > 1).count()
mask

np.int64(147)

147 кабин, в которых было от 2 и более человек